# LitWBTransferLearningResnet notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TXSTCODEPLAYGROUND/CodeCS4337Fall2026/blob/main/LitWBTransferLearningResnet/lit_wb_transfer_learning_resnet_notebook.ipynb)

This notebook trains **LitWBTransferLearningResnet**: a ResNet-18 pretrained on ImageNet, transferred to Oxford Flowers-102 (102 species, 10 training images per class). What the project does and what it teaches is in its [README](README.md). Every project has its own notebook like this one, in its folder.

The project has **three configs**, one per transfer-learning strategy. This notebook runs all three and compares them:

| Config | Strategy | What is trained |
| --- | --- | --- |
| `config01.json` | Feature extraction | Only the new head (52,326 parameters); the pretrained backbone is frozen |
| `config02.json` | Fine-tuning | Everything; the backbone with a 10 times smaller learning rate |
| `config03.json` | From scratch | Everything, from random weights: the baseline |

### Choose your route

| Route | Sections to run | Then run the project with |
| --- | --- | --- |
| **Colab, from notebook cells** | 1 (setup), then 2 and 3 (4 and 5 to load or keep training a model) | Option A1 (`!python -m ...`) or Option B (`main(...)`) |
| **Colab, from Colab's terminal** (Colab Pro) | 1 (setup) in the notebook first | Option A2: the same commands in the terminal |
| **Locally, from this notebook** | Skip 1, start at [**2. Run the project**](#2.-Run-the-project-(choose-one-option)) | Option A1 or B, with the `.venv` as the notebook's kernel |
| **Locally, from your terminal** | None, no notebook needed | `python -m LitWBTransferLearningResnet --config config01.json` from the repo root, with the `.venv` activated |
| **Colab CLI, for the final training run** | None, no notebook needed | `python -m LitWBTransferLearningResnet --config config01.json` on a Colab runtime opened from your terminal: see [**Training with the Colab CLI**](#Training-with-the-Colab-CLI) |

In Colab, the setup cells are needed even when you train in the terminal: Google Drive can only be mounted, and Colab Secrets only read, from a notebook cell. Locally, set up the repo once first (`.venv`, requirements, and `.env`, see the main README), and open this notebook from the project folder.

In Colab, run the cells from top to bottom.

1. **Use a GPU:** *Runtime → Change runtime type → GPU*. This project needs one: on a CPU, every evaluation of the 6,149 test images takes minutes.
2. **Mount Google Drive** so your results survive runtime resets.
3. **Set up the repo:** clone (or update) it, apply the Colab settings, and install only the missing requirements.
4. **Load your API keys** (optional) from Colab Secrets.
5. **Run the three configs**, either as terminal commands or from Python, and compare the results.
6. **Load a trained model** later, without training again (section 4).
7. **Keep training** a run for more epochs, or finish one cut off by a disconnect (section 5).

Before a long training run, read the [pro tips](#pro-tips-keep-training-running) at the end.

## 1. Setup

> **Running locally?** Skip this section: these cells only work in Colab. Jump to [**2. Run the project**](#2.-Run-the-project-(choose-one-option)).

Run these cells once per session (and again after the runtime resets): mount Drive, set up the repo, change directory, and load API keys. The setup cell is safe to re-run, e.g. to get new projects with `git pull`; run the API-key cell again after it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
set -e

cd /content
if [ -d CodeCS4337Fall2026/.git ]; then
  git -C CodeCS4337Fall2026 pull --ff-only
else
  git clone https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git
fi
cd CodeCS4337Fall2026

# Colab settings: data on the local disk, results in Google Drive.
cp .envcolab .env

# Install only packages Colab does not already have (versions are not pinned).
grep -vE '^\s*(#|$)' requirements.txt \
  | sed -E 's/[<>=!~;[ ].*//' \
  | while read -r pkg; do
      if pip show "$pkg" > /dev/null 2>&1; then
        echo "skip $pkg (already installed)"
      else
        pip install -q "$pkg" && echo "installed $pkg"
      fi
    done

In [ ]:
# The bash cell above cannot change the notebook's working directory, so do it here.
%cd /content/CodeCS4337Fall2026

### Load API keys (optional)

This project logs every run to Weights & Biases (W&B), which needs `WANDB_API_KEY`. Without it, runs are logged to W&B offline and can be uploaded later with the `wandb sync` command printed at the end of the run.

1. Add the keys as Colab **Secrets**: key icon in the left sidebar → *Add new secret*, named `WANDB_API_KEY`. Never type keys into a notebook cell or into `.envcolab`, which is public on GitHub.
2. Run the cell below **after the setup cells and before running the project**, whether you run it from this notebook (Option A1 or B) or from Colab's terminal (A2). Run it again whenever you re-run the setup cell.

The cell loads `.env`, then looks up every key that is still missing in your Secrets (Colab may ask you to *Grant access*). Keys it finds are added to this notebook's environment and saved to `/content/CodeCS4337Fall2026/.env`, so terminal runs find them too. That file is not in git and is deleted when the runtime resets. Values are never printed.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google.colab import userdata

ENV_FILE = Path("/content/CodeCS4337Fall2026/.env")
KEYS = ["WANDB_API_KEY"]

if load_dotenv(ENV_FILE):
    print(f"Loaded settings from {ENV_FILE}")
else:
    print(f"No settings in {ENV_FILE}: run the setup cells first")

for key in KEYS:
    if os.getenv(key):
        print(f"{key}: found in .env")
        continue
    try:
        os.environ[key] = userdata.get(key)
    except Exception:  # noqa: BLE001 (Secret not added or access not granted)
        print(f"{key}: not found, W&B will log offline")
        continue
    with ENV_FILE.open("a") as env_file:
        env_file.write(f"\n{key}={os.environ[key]}\n")
    print(f"{key}: loaded from Colab Secrets and saved to .env")

## 2. Run the project (choose one option)

Both options run exactly the same training and save results to the same folder (Google Drive in Colab, `runs/` locally). Pass a config name from the project's `configs/` folder. The three configs are `config01.json` (frozen backbone), `config02.json` (fine-tuning), and `config03.json` (from scratch). To try your own settings, copy one of them in the file browser (left sidebar, `CodeCS4337Fall2026/LitWBTransferLearningResnet/configs/`), e.g. to `config04.json`, edit it, and pass its name.

**Run the next cell first** (in Colab and locally): it finds the repo folder and makes the project importable.

| Option | How | When to use it |
| --- | --- | --- |
| **A. Terminal command** | `!` in a notebook cell, or Colab's terminal (Colab Pro) | Same command as on your own machine |
| **B. From Python** | `from LitWBTransferLearningResnet import main` | Simple, everything stays in the notebook |

In [ ]:
import sys
from pathlib import Path

# Colab: the clone made by the setup cell. Locally: the parent of this notebook's project folder.
IN_COLAB = "google.colab" in sys.modules
REPO = Path("/content/CodeCS4337Fall2026") if IN_COLAB else Path.cwd().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(f"Repo folder: {REPO}")

### Where your results go

Run the cell below and look at what it prints: `OUTPUT_DIR` is the folder where **every run of this project is saved**: checkpoints, metrics, and `runs_summary.csv`. It is set in `.env` at the repo root.

- **In Colab** it is in your Google Drive, `/content/drive/MyDrive/CodeCS4337Fall2026/runs` (from `.envcolab`), so your results survive runtime resets. Open *MyDrive → CodeCS4337Fall2026 → runs* in Google Drive to find them later.
- **Locally** it is `runs`, i.e. the `runs/` folder of the repo (relative paths are relative to the repo root).

Each run gets its own folder, `<OUTPUT_DIR>/LitWBTransferLearningResnet/<config>/<timestamp>/`.

In [ ]:
import os

from dotenv import load_dotenv

# Colab: the copy of .envcolab made by the setup cell. Locally: your own .env.
load_dotenv(REPO / ".env")
!echo "OUTPUT_DIR=$OUTPUT_DIR"
output_dir = REPO / os.getenv("OUTPUT_DIR", "runs")
print(
    f"Runs of this project are saved in: {output_dir / 'LitWBTransferLearningResnet'}"
)

### Option A: Run as a terminal command

**A1. From a notebook cell (everyone):** a line starting with `!` is sent to the shell, and `{REPO}` is replaced by the repo folder found above. Run the cell below. Locally, if it reports a missing package, the shell is not using the `.venv`'s Python: use Option B, or run the same command in your own terminal with the `.venv` activated.

In [ ]:
!cd "{REPO}" && for c in config01 config02 config03; do python -m LitWBTransferLearningResnet --config $c.json; done

**A2. From Colab's terminal (Colab Pro):** open the terminal from the **Terminal** button at the bottom left of the Colab window, then run:

```bash
cd /content/CodeCS4337Fall2026
python -m LitWBTransferLearningResnet --config config01.json
python -m LitWBTransferLearningResnet --config config02.json
python -m LitWBTransferLearningResnet --config config03.json
```

The terminal runs on the same machine as this notebook, so it uses the same files, packages, and Drive mount. **You still need to run the setup cells above first**: Drive can only be mounted from a notebook cell, and Colab Secrets can only be read from a notebook cell (the API-key cell saves them to `.env`, where the terminal finds them). While training runs in the terminal, the notebook stays free for other work.

### Option B: Run from Python

The project can be imported because the cell at the start of this section added the repo folder to Python's import path. This works in Colab and locally. Each call trains one config, evaluates its best checkpoint on the validation and test sets, and returns the test metrics; run a single config by keeping only its name in the list.

In [ ]:
from LitWBTransferLearningResnet import main

test_metrics = {}
for config in ["config01.json", "config02.json", "config03.json"]:
    test_metrics[config] = main(config)

## 3. Results

Each run is saved under `<OUTPUT_DIR>/LitWBTransferLearningResnet/<config>/<timestamp>/`: in Colab that is Google Drive, `MyDrive/CodeCS4337Fall2026/runs/`; locally, the `runs/` folder of the repo (both set by `OUTPUT_DIR` in `.env`). Each config has a `runs_summary.csv` with one row per run.

The cell below compares the **newest run of each config**: how many parameters were trained, how many epochs it ran (early stopping), and its test scores. `test_f1` is the macro F1 (every class counts the same) and `test_acc_top5` how often the true species is among the 5 highest scores.

In [ ]:
import csv
import os

from dotenv import load_dotenv

load_dotenv(REPO / ".env")
# OUTPUT_DIR from .env: absolute in Colab (Drive), relative to the repo locally.
project_runs = REPO / os.getenv("OUTPUT_DIR", "runs") / "LitWBTransferLearningResnet"

columns = [
    "trainable_params",
    "epochs_run",
    "best_val_acc",
    "test_acc",
    "test_acc_top5",
    "test_f1",
]
print(f"{'config':<10}" + "".join(f"{c:>18}" for c in columns))
for config in ["config01", "config02", "config03"]:
    summary = project_runs / config / "runs_summary.csv"
    if not summary.exists():
        print(f"{config:<10}  (no run yet)")
        continue
    with summary.open() as f:
        newest = list(csv.DictReader(f))[-1]
    print(f"{config:<10}" + "".join(f"{newest[c]:>18}" for c in columns))

### In W&B

Open the W&B project `LitWBTransferLearningResnet` (the link is printed at the end of each run). The runs are grouped by config.

- **Learning curves:** add line plots with `epoch` as X and `train_acc`, `val_acc` as Y: the two pretrained runs start far above the run from scratch.
- **Evaluation report**, logged once per run for the best checkpoint, for the validation (`val_...`) and test (`test_...`) sets:
  - `<split>_predictions`: filter with `row["correct"] = false` to see only the mistakes;
  - `<split>_correct_examples` and `<split>_wrong_examples` (the most confident mistakes), captioned *true / predicted (confidence)*;
  - `<split>_per_class`: sort by `f1` to find the hardest species;
  - `<split>_pr_curve`: precision-recall curves of the 5 worst and 5 best classes;
  - `<split>_confusion_matrix` and `<split>_most_confused`: which species get mixed up.

How to read each of them is explained in the [README](README.md#reading-the-report).

## 4. Load a trained model

Running a config again trains a **new** model from scratch, unless you resume an earlier run (section 5). To use a model you already trained (after a runtime reset, to test it again, or to try it on images), load it with `load_model`: it finds the checkpoint under `<OUTPUT_DIR>/LitWBTransferLearningResnet/`, rebuilds the network with that run's settings, and loads its weights. Nothing is trained.

**Before this section:** you need at least one run of the config, trained in section 2 (in Colab: trained with Drive mounted, so it was saved to Drive). Then, in Colab, run the setup cells of section 1 (Drive must be mounted, that is where your runs are); in Colab and locally, run the first cell of section 2, which finds the repo folder.

| Call | What it loads |
| --- | --- |
| `load_model("config01")` | The newest run of `config01`, its best checkpoint (`best_*.ckpt`) |
| `load_model("config01/2026-10-02_12-29-09")` | One specific run: the folder name of the run, under `LitWBTransferLearningResnet/config01/` |
| `load_model("config01", which="last")` | The last epoch instead of the best one |
| `load_model("config01/2026-10-02_12-29-09/best_epochNN_valaccX.XXXX.ckpt")` | One specific checkpoint file of a run |
| `load_model("config02")` | The newest run of `config02` (fine-tuning) |

Each run folder keeps **two checkpoints**: the best epoch (`best_*.ckpt`) and the last epoch (`last.ckpt`); older ones are replaced during training. The model comes back ready for predictions (in eval mode, on the CPU; add `device="cuda"` for the GPU). It prints which checkpoint it loaded.

In [ ]:
from LitWBTransferLearningResnet import load_model

model = load_model("config02")

Try the loaded model on test images: the cell below shows 8 random test images with the true and predicted species and the confidence (red when wrong), then the accuracy on all 6,149 test images. Images get the same evaluation preprocessing as in training. The accuracy should match the run's `test_acc` in `runs_summary.csv` when you loaded the best checkpoint. On a CPU, the full test set takes a few minutes.

In [ ]:
import os

import matplotlib.pyplot as plt
import torch
from dotenv import load_dotenv
from torch.utils.data import DataLoader

from LitWBTransferLearningResnet.dataloaders import (
    FLOWERS102_CLASSES,
    IMAGENET_MEAN,
    IMAGENET_STD,
    Flowers102DataModule,
)

load_dotenv(REPO / ".env")
data = Flowers102DataModule(str(REPO / os.getenv("DATA_DIR", "data")))
data.prepare_data()
data.setup("test")
test_set = data.test_set

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

indices = torch.randperm(len(test_set), generator=torch.Generator().manual_seed(0))[:8]
images, labels = zip(*(test_set[i] for i in indices.tolist()))
with torch.no_grad():
    probs = model(torch.stack(images).to(device)).softmax(dim=1).cpu()
confidence, preds = probs.max(dim=1)

fig, axes = plt.subplots(2, 4, figsize=(14, 7.5), layout="constrained")
for ax, image, label, pred, conf in zip(axes.flat, images, labels, preds, confidence):
    ax.imshow((image * std + mean).clamp(0, 1).permute(1, 2, 0))
    ax.set_title(
        f"true: {FLOWERS102_CLASSES[label]}\npred: {FLOWERS102_CLASSES[pred]} ({conf:.0%})",
        color="black" if pred == label else "red",
        fontsize=9,
    )
    ax.axis("off")
plt.show()

correct = 0
with torch.no_grad():
    for x, y in DataLoader(test_set, batch_size=128, num_workers=2):
        correct += (model(x.to(device)).argmax(dim=1).cpu() == y).sum().item()
print(f"Test accuracy: {correct / len(test_set):.4f}")

## 5. Keep training a run

`load_model` gives you a trained model to use; it does not train it further. To **continue training** an earlier run, for more epochs or to finish a run cut off by a Colab disconnect, pass `resume_from` to `main`. Training continues from the run's last checkpoint (`last.ckpt`, saved after every epoch) and runs the config's `"epochs"` **more** epochs: a run that trained 12 epochs, resumed with `config02.json` (30 epochs), can go up to epoch 42, or earlier if early stopping stops it.

**Before this section:** the same as for section 4 (a saved run; in Colab, section 1 with Drive mounted; then the first cell of section 2).

| Call | What it continues |
| --- | --- |
| `main("config02.json", resume_from="config02")` | The newest run of `config02`, for `"epochs"` more epochs. Run it again to continue further: the newest run is then the continued one. |
| `main("config02.json", resume_from="config02/2026-10-02_12-29-09")` | One specific run (its folder name) |
| `main("config02.json", resume_from="config02/2026-10-02_12-29-09/last.ckpt")` | One specific checkpoint file |
| `main("config02_more.json", resume_from="config02")` | A different number of extra epochs: a copy of the config with another `"epochs"` (saved under `config02_more/`) |

What comes from where:

- **From the checkpoint:** the weights, the epoch count, and the optimizer state, including its **learning rates** and weight decay. Changing `lr`, `backbone_lr`, or `weight_decay` in the config has no effect when resuming.
- **From the config:** the number of extra `"epochs"`, the data settings (e.g. `batch_size`), the seed, and logging. The **model settings must match the run's**, in particular `freeze_backbone`, which decides what the optimizer trains: the saved weights only fit the same network.
- **Early stopping** starts counting again: a run that stopped early can be resumed, and gets the config's `"patience"` epochs to beat the **earlier run's best** `val_acc`.
- The continued run is saved as a **new run folder**, with `resumed_from` in its `hparams.json`, and gets its own row in `runs_summary.csv`. Its best checkpoint and test score are those of the continued run. The earlier run is not changed.

The same from a terminal: `python -m LitWBTransferLearningResnet --config config02.json --resume-from config02`.

In [ ]:
from LitWBTransferLearningResnet import main

# The config's "epochs" more epochs, on top of the newest config02 run.
main("config02.json", resume_from="config02")

## Training with the Colab CLI

Another way to train this project, once everything is final. Its only purpose
is the training run itself:

1. **Work locally** on the project: change the code or a config, run short
   tests, fix bugs.
2. **Use Google Colab** (this notebook) for debugging and interactive work.
3. **Use the Colab CLI** when everything is final and the only thing left is
   training. The CLI creates a Colab runtime from your local terminal and opens
   a shell on it, so there is no notebook or browser tab to keep open. On the
   runtime you clone the repo and run `python -m LitWBTransferLearningResnet --config config01.json`.

The Colab CLI works on Linux and macOS (not Windows). For more detail, see
[google-colab-cli.md](https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026/blob/main/google-colab-cli.md) at the repo root or the
[official Colab CLI documentation](https://github.com/googlecolab/google-colab-cli).

### Step 1: Install the CLI

On your own machine:

```bash
uv tool install google-colab-cli     # or: pip install google-colab-cli
colab version                         # check that it works
```

The first command that talks to Colab prints a sign-in link: open it, sign in
with your Google account, and paste the code it shows back into the terminal.

### Step 2: The basic workflow (sessions)

A **session** is one Colab runtime (a virtual machine) with a name you choose,
here `cs4337`. These commands run in your **local** terminal:

| What | Command |
| --- | --- |
| Create a session with a GPU | `colab new -s cs4337 --gpu T4` (or `L4`, `A100`, `H100`, depending on your plan) |
| Create a session without a GPU | `colab new -s cs4337` (CPU only) |
| List your sessions / check one | `colab sessions` / `colab status -s cs4337` |
| Attach to a session (open a shell on it) | `colab ssh -s cs4337` |
| Leave the shell; the session keeps running | `exit` |
| Stop the session (delete the runtime) | `colab stop -s cs4337` |

Use a GPU runtime: `--gpu T4` works, and `--gpu L4` or `--gpu A100` is faster if your Colab plan has them. A CPU session can't be switched to a GPU: stop it and create a new one.

> **Once a session is stopped, nothing on it stays.** The cloned repo, the
> downloaded data, and every file that is not on Google Drive are deleted.
> This project writes its runs to Google Drive (step 4), so mount Drive before
> training.

### Step 3: Mount Google Drive

Drive is mounted from your local terminal, not from inside the session. If you
are attached, `exit` first:

```bash
exit                                # only if you are inside the session
colab drivemount -s cs4337          # open the link it prints and allow access
colab ssh -s cs4337                 # attach again
ls /content/drive/MyDrive           # check: your Drive files are listed
```

### Step 4: Clone the repo and train

Inside the session, clone the repo and go to its root folder:

```bash
cd /content
git clone https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git
cd CodeCS4337Fall2026
cp .envcolab .env
```

The runtime only sees what is on GitHub: if you changed code or a config
locally, push it to your own fork first and clone that instead. `.envcolab`
sets `DATA_DIR=/content/data` (data on the runtime's fast local disk) and
`OUTPUT_DIR=/content/drive/MyDrive/CodeCS4337Fall2026/runs` (runs on Drive, so they are kept after a stop).

This project logs to Weights & Biases and reads the key only from `.env`.
Add your W&B API key (from [wandb.ai/authorize](https://wandb.ai/authorize))
to the runtime's `.env`. `read -s` hides the key while you paste it, so it is
not shown on screen or saved in the shell history:

```bash
read -rsp "W&B API key: " key && echo "WANDB_API_KEY=$key" >> .env && unset key && echo
```

Without a key the run still trains, but W&B logs offline into the run folder;
upload it later with the `wandb sync` command that the run prints. Never put
the key in `.envcolab`, which is in git.

Install only the packages Colab doesn't already have, as the setup of this notebook
does. A plain `pip install -r requirements.txt` would replace Colab's own
versions, which is slow and can break other preinstalled packages:

```bash
grep -vE '^\s*(#|$)' requirements.txt \
  | sed -E 's/[<>=!~;[ ].*//' \
  | while read -r pkg; do
      if pip show "$pkg" > /dev/null 2>&1; then echo "skip $pkg (already installed)"
      else pip install -q "$pkg" && echo "installed $pkg"; fi
    done
```

Start a `tmux` session first, so training keeps going if the SSH connection
drops or you close your laptop, then run the training from the repo root:

```bash
tmux new -s train                  # if tmux is missing: apt-get install -y tmux
python -m LitWBTransferLearningResnet --config config01.json
```

To train all 3 configs one after the other, in the same `tmux` session:

```bash
for c in config01 config02 config03; do python -m LitWBTransferLearningResnet --config "$c.json"; done
```

To leave training running, press `Ctrl+B`, then `D`. To come back later:
`colab ssh -s cs4337`, then `tmux attach -t train`.

### Step 5: Check that everything was logged

The run folders are on Google Drive, so they are kept after the session
stops. From the session:

```bash
ls /content/drive/MyDrive/CodeCS4337Fall2026/runs/LitWBTransferLearningResnet/config01
```

or on [drive.google.com](https://drive.google.com), under **My Drive >
CodeCS4337Fall2026 > runs > LitWBTransferLearningResnet**. Check that:

- each run has its own `<timestamp>` folder (the path is printed when the run starts), with its config, results, plots, and checkpoints;
- `runs_summary.csv`, next to the run folders, has one new row per run;
- the run shows up in your W&B project: the run prints its W&B link. If it printed a `wandb sync` command instead, the run was logged offline (no key or no login); run that command later, from a machine with your key, to upload it;
- if the session stopped or the run was cut off, start a new session, repeat the steps above, and continue from the last checkpoint on Drive with `--resume-from config01`, e.g. `python -m LitWBTransferLearningResnet --config config01.json --resume-from config01`.

When everything is checked, stop the session so it no longer uses your Colab
quota:

```bash
colab stop -s cs4337
```

## Pro tips: keep training running

Colab disconnects runtimes that look idle (no interaction for a while, often around 90 minutes) and also caps how long a runtime can run in total (around 12 hours on the free tier, longer on paid plans). The exact limits vary and are not guaranteed. When the runtime disconnects, training stops and everything under `/content/` (outside Drive) is deleted.

**Save your state in Google Drive**
- Always mount Drive *before* training. Results then go straight to Drive (`OUTPUT_DIR` in `.envcolab`), not to the temporary disk.
- Two checkpoints are saved to Drive *during* training: the best model so far (`best_epochXX_valaccY.ckpt`), every time validation accuracy improves, and the last epoch (`last.ckpt`), after every epoch. If the runtime disconnects mid-run, both are kept.
- The run's other files (results, plots, and the row in `runs_summary.csv`) are written only when the run finishes. To finish an interrupted run, resume it (section 5): it continues from the last saved epoch.
- Keep your notebook in Drive too: *File → Save a copy in Drive*. Keep copies of configs you edited in Drive, since `/content/CodeCS4337Fall2026` is deleted on reset.

**Avoid idle disconnects**
- Keep the Colab browser tab open, and don't let your computer go to sleep (plug in your laptop and turn off sleep while training).
- Check on the notebook now and then (scroll, look at the progress bars). Running in the terminal does *not* prevent disconnects: the runtime is shared.
- Don't use scripts or browser extensions that fake activity (auto-clickers); they can get your Colab usage restricted.
- With Colab Pro+, *background execution* keeps the runtime alive even after you close the browser.

**Plan your runs**
- Test a new config with few epochs first (e.g. `"epochs": 1`), then start the full run.
- Run the three configs one at a time if your session is short: each `main(...)` call saves its own results, and `config03` (from scratch) takes the longest.
- Prefer several shorter runs over one very long run; each run saves its own results.
- After a disconnect, re-run the setup cells (including the API-key cell), then resume the run with `resume_from` (section 5) instead of starting over.
- When you are done, free the GPU with *Runtime → Disconnect and delete runtime*. It saves your GPU quota (or compute units on paid plans) for next time.